# Heavy interactive workloads → `lh_fabric_management`

Ranks **reports and semantic models by the interactive CU they burn**, so you can
prioritize what to move into the warehouse. Source is the **Microsoft Fabric
Capacity Metrics** app model, table `Metrics By Item And Operation`.

Interactive vs background isn't a column in this app build — it's implied by the
**operation name** (`Query`, `Render`, `XMLA Read`, `SQL Endpoint Query` = interactive;
`Dataset Scheduled Refresh`, `Notebook Scheduled Run`, `DataMovement` = background).
The `INTERACTIVE_OPS` set below is the classifier — tweak it to taste.

Writes `gold_heavy_interactive_items` (one row per capacity×item: total / interactive /
background CU, operations, top operation) to the `fabricmanagement` schema.

## Prerequisites
- **Microsoft Fabric Capacity Metrics** app installed; run identity has read + **write/owner**
  (the facts are DirectQuery-gated, so we bind `DefaultCapacityID` + window per capacity).
- Tenant setting **"Semantic Model Execute Queries REST API"** ON.

In [ ]:
# ── Config & auth ────────────────────────────────────────────────────────────
from datetime import datetime, timezone, timedelta
from collections import Counter
import time
import requests
import notebookutils

POWERBI_API      = "https://api.powerbi.com/v1.0/myorg"
LAKEHOUSE_NAME   = "lh_fabric_management"
LAKEHOUSE_SCHEMA = "fabricmanagement"
TBL_HEAVY        = "gold_heavy_interactive_items"

WINDOW_DAYS        = 14
METRICS_DATASET_ID = None      # set to skip auto-discovery
METRICS_GROUP_ID   = None
CAP_PARAM, START_PARAM, END_PARAM = "DefaultCapacityID", "StartDate", "EndDate"

# Operations counted as interactive (user-initiated). Everything else = background.
INTERACTIVE_OPS = {
    "Query", "Render", "XMLA Read Operation", "SQL Endpoint Query", "Warehouse Query",
    "Dataset On-Demand Refresh", "Web Modeling Read Operation", "Web Modeling Write Operation",
    "XMLA Write Operation", "Copilot In Fabric", "AI Function Evaluation",
    "High Concurrency Session Livy Run", "Notebook Run",
}
# Kinds to highlight in the verify step (reports + semantic models). Gold keeps all kinds.
FOCUS_KINDS = ("Report", "PaginatedReport", "Dataset", "Model", "Datamart")

RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
print("Run timestamp (UTC):", RUN_TS.isoformat())

_TOKEN = {"v": None, "exp": 0.0}
def _headers():
    if time.time() > _TOKEN["exp"]:
        _TOKEN["v"] = notebookutils.credentials.getToken("pbi")
        _TOKEN["exp"] = time.time() + 3000
    return {"Authorization": f"Bearer {_TOKEN['v']}", "Content-Type": "application/json"}

def _get(url):
    r = requests.get(url, headers=_headers(), timeout=120); r.raise_for_status(); return r.json()

def _route(suffix):
    if METRICS_GROUP_ID:
        return f"{POWERBI_API}/groups/{METRICS_GROUP_ID}/datasets/{METRICS_DATASET_ID}/{suffix}"
    return f"{POWERBI_API}/datasets/{METRICS_DATASET_ID}/{suffix}"

def dax(query):
    r = requests.post(_route("executeQueries"), headers=_headers(),
                      json={"queries": [{"query": query}], "serializerSettings": {"includeNulls": True}},
                      timeout=180)
    if not r.ok:
        raise RuntimeError(f"executeQueries {r.status_code}: {r.text[:400]}")
    t = r.json()["results"][0]["tables"]
    return [{k.strip("[]"): v for k, v in row.items()} for row in (t[0]["rows"] if t else [])]

def bind_params(updates):
    r = requests.post(_route("Default.UpdateParameters"), headers=_headers(),
                      json={"updateDetails": [{"name": k, "newValue": v} for k, v in updates.items()]},
                      timeout=60)
    if not r.ok:
        raise RuntimeError(f"UpdateParameters {r.status_code}: {r.text[:300]}")

def _tables_path(name):
    lh = notebookutils.lakehouse.get(name)
    props = (lh.get("properties") or {}) if isinstance(lh, dict) else {}
    return (props.get("oneLakeTablesPath") or f'{props.get("abfsPath", "").rstrip("/")}/Tables')

TABLES_PATH = _tables_path(LAKEHOUSE_NAME)
def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

In [ ]:
# ── 1. Discover the Capacity Metrics semantic model ──────────────────────────
if not METRICS_DATASET_ID:
    print("Searching workspaces for the Capacity Metrics model...")
    for g in _get(f"{POWERBI_API}/groups").get("value", []):
        try:
            for d in _get(f"{POWERBI_API}/groups/{g['id']}/datasets").get("value", []):
                if "capacity metrics" in (d.get("name") or "").lower():
                    METRICS_DATASET_ID, METRICS_GROUP_ID = d["id"], g["id"]
                    print(f"  found '{d.get('name')}'  dataset={d['id']}  workspace='{g.get('name')}'")
                    break
        except requests.HTTPError:
            continue
        if METRICS_DATASET_ID:
            break
if not METRICS_DATASET_ID:
    raise RuntimeError("Capacity Metrics model not found -- set METRICS_DATASET_ID/METRICS_GROUP_ID.")
print("Using dataset:", METRICS_DATASET_ID, "| workspace:", METRICS_GROUP_ID)

In [ ]:
# ── 2. Pull item x operation CU per capacity (facts are gated -> bind window) ─
OP_DAX = "\n".join([
    "DEFINE",
    "  VAR __g = GROUPBY('Metrics By Item And Operation',",
    "    'Metrics By Item And Operation'[Capacity Id],",
    "    'Metrics By Item And Operation'[Item Id],",
    "    'Metrics By Item And Operation'[Artifact kind],",
    "    'Metrics By Item And Operation'[Operation name],",
    "    \"CU_s\", SUMX(CURRENTGROUP(), 'Metrics By Item And Operation'[CU (s)]),",
    "    \"Ops\",  SUMX(CURRENTGROUP(), 'Metrics By Item And Operation'[Operations]) )",
    "EVALUATE",
    "  SELECTCOLUMNS( ADDCOLUMNS( __g,",
    "      \"item_name\",     COALESCE(MAXX(FILTER(ALL('Items'), 'Items'[Item Id] = 'Metrics By Item And Operation'[Item Id]), 'Items'[Item name]), 'Metrics By Item And Operation'[Item Id]),",
    "      \"workspace\",     MAXX(FILTER(ALL('Items'), 'Items'[Item Id] = 'Metrics By Item And Operation'[Item Id]), 'Items'[Workspace name]),",
    "      \"capacity_name\", MAXX(FILTER(ALL('Capacities'), 'Capacities'[Capacity Id] = 'Metrics By Item And Operation'[Capacity Id]), 'Capacities'[Capacity name]) ),",
    "    \"capacity_id\",   'Metrics By Item And Operation'[Capacity Id],",
    "    \"item_id\",       'Metrics By Item And Operation'[Item Id],",
    "    \"artifact_kind\", 'Metrics By Item And Operation'[Artifact kind],",
    "    \"operation\",     'Metrics By Item And Operation'[Operation name],",
    "    \"item_name\",     [item_name],",
    "    \"workspace\",     [workspace],",
    "    \"capacity_name\", [capacity_name],",
    "    \"cu_s\",          [CU_s],",
    "    \"operations\",    [Ops] )",
])
CAPS_DAX = ("EVALUATE SELECTCOLUMNS('Capacities', \"capacity_id\", 'Capacities'[Capacity Id], "
            "\"capacity_name\", 'Capacities'[Capacity name])")

start = (RUN_TS - timedelta(days=WINDOW_DAYS + 1)).strftime("%Y-%m-%dT00:00:00")
end   = (RUN_TS + timedelta(days=1)).strftime("%Y-%m-%dT00:00:00")
caps  = dax(CAPS_DAX)
print(f"Capacities: {[c.get('capacity_name') for c in caps]}")

raw = []
for c in caps:
    cid = c.get("capacity_id")
    if not cid:
        continue
    try:
        bind_params({CAP_PARAM: cid, START_PARAM: start, END_PARAM: end})
        r = dax(OP_DAX)
        raw.extend(r)
        print(f"  {c.get('capacity_name')}: {len(r)} item-operations")
    except RuntimeError as exc:
        print(f"  {c.get('capacity_name')}: {exc}")

# roll up per (capacity, item); classify interactive vs background by operation name
agg = {}
for r in raw:
    key = (r.get("capacity_id"), r.get("item_id"))
    a = agg.setdefault(key, {
        "capacity_id": r.get("capacity_id"), "capacity_name": r.get("capacity_name"),
        "workspace": r.get("workspace"), "item_id": r.get("item_id"),
        "item_name": r.get("item_name"), "artifact_kind": r.get("artifact_kind"),
        "total_cu_s": 0.0, "interactive_cu_s": 0.0, "background_cu_s": 0.0,
        "interactive_ops": 0, "total_ops": 0, "_byop": {}})
    cu = float(r.get("cu_s") or 0); ops = int(r.get("operations") or 0); op = r.get("operation")
    a["total_cu_s"] += cu; a["total_ops"] += ops
    if op in INTERACTIVE_OPS:
        a["interactive_cu_s"] += cu; a["interactive_ops"] += ops
    else:
        a["background_cu_s"] += cu
    a["_byop"][op] = a["_byop"].get(op, 0.0) + cu

heavy_rows = []
for a in agg.values():
    top_op = max(a["_byop"], key=a["_byop"].get) if a["_byop"] else None
    heavy_rows.append({
        "capacity_id": a["capacity_id"], "capacity_name": a["capacity_name"],
        "workspace": a["workspace"], "item_id": a["item_id"], "item_name": a["item_name"],
        "artifact_kind": a["artifact_kind"],
        "total_cu_s": round(a["total_cu_s"], 2),
        "interactive_cu_s": round(a["interactive_cu_s"], 2),
        "background_cu_s": round(a["background_cu_s"], 2),
        "interactive_ops": a["interactive_ops"], "total_ops": a["total_ops"],
        "top_operation": top_op, "window_days": WINDOW_DAYS, "run_timestamp": RUN_TS,
    })
print(f"\nItems: {len(heavy_rows)}")
for k, n in Counter(x["artifact_kind"] for x in heavy_rows).most_common():
    print(f"  {str(k):<18} {n}")

In [ ]:
# ── 3. Write gold_heavy_interactive_items ────────────────────────────────────
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, TimestampType)

schema = StructType([
    StructField("capacity_id", StringType()),
    StructField("capacity_name", StringType()),
    StructField("workspace", StringType()),
    StructField("item_id", StringType()),
    StructField("item_name", StringType()),
    StructField("artifact_kind", StringType()),
    StructField("total_cu_s", DoubleType()),
    StructField("interactive_cu_s", DoubleType()),
    StructField("background_cu_s", DoubleType()),
    StructField("interactive_ops", IntegerType()),
    StructField("total_ops", IntegerType()),
    StructField("top_operation", StringType()),
    StructField("window_days", IntegerType()),
    StructField("run_timestamp", TimestampType()),
])
names = [f.name for f in schema.fields]
df = spark.createDataFrame([tuple(d.get(n) for n in names) for d in heavy_rows], schema=schema)
(df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(_table_uri(TBL_HEAVY)))
print(f"  {df.count()} rows -> {TBL_HEAVY}")

## Verify — heaviest interactive reports & semantic models

In [ ]:
spark.read.format("delta").load(_table_uri(TBL_HEAVY)).createOrReplaceTempView("heavy")
kinds = "', '".join(["Report", "PaginatedReport", "Dataset", "Model", "Datamart"])

print("Top 20 reports / semantic models by interactive CU:")
display(spark.sql(
    f"SELECT artifact_kind, item_name, workspace, capacity_name, "
    f"       interactive_cu_s, total_cu_s, interactive_ops, top_operation "
    f"FROM heavy WHERE artifact_kind IN ('{kinds}') "
    f"ORDER BY interactive_cu_s DESC LIMIT 20"))

print("Interactive CU by artifact kind (all items):")
display(spark.sql(
    "SELECT artifact_kind, COUNT(*) AS items, "
    "       ROUND(SUM(interactive_cu_s),0) AS interactive_cu_s, "
    "       ROUND(SUM(total_cu_s),0) AS total_cu_s "
    "FROM heavy GROUP BY artifact_kind ORDER BY interactive_cu_s DESC"))